# فصل ۱۳: پردازش داده‌ها در LangChain (Data Processing)
## بخش ۲: بارگیری اسناد از فرمت‌های گوناگون (Document Loaders)

داده‌های واقعی در دنیای سازمانی در فرمت ساده متنی (`.txt`) نیستند! اطلاعات ارزشمند کسب‌وکارها در فایل‌های **PDF**، صفحات **وب** پر از تبلیغات و منوهای مزاحم، مستندات **Word** و شیت‌های **Excel** قرار دارند.

در فریم‌ورک **LangChain** بیش از ۸۰ نوع لودر مختلف توسعه یافته است که همگی از یک استاندارد واحد پیروی می‌کنند: **تبدیل هر نوع داده به لیستی از اشیاء `Document`**.

در این بخش به بررسی عملی ۳ لودر پرکاربرد (**TextLoader**، **PyPDFLoader** و **WebBaseLoader**) و یک تکنیک حیاتی مهندسی نرم‌افزار به نام **Lazy Loading** می‌پردازیم.

### ۱. بارگذاری فایل‌های متنی و مارک‌داون با `TextLoader`

ساده‌ترین روش برای خواندن فایل‌های متنی (`.txt` یا `.md`) استفاده از `TextLoader` است.

> **نکته برای متون فارسی:** برای جلوگیری از خطای `UnicodeDecodeError` در سیستم‌عامل ویندوز، همواره آرگومان **`encoding="utf-8"`** را به لودر بدهید.

In [ ]:
from langchain_community.document_loaders import TextLoader

# بارگذاری فایل متنی سخنرانی تاریخی 'من رویایی دارم'
txt_path = "I_have_a_dream_fa.txt"
txt_loader = TextLoader(txt_path, encoding="utf-8")

txt_docs = txt_loader.load()

print(f"تعداد اسناد بارگذاری شده: {len(txt_docs)}")
print(f"متادیتای سند: {txt_docs[0].metadata}")
print("\nبخشی از محتوای سند:")
print(txt_docs[0].page_content[:200] + "...")

### ۲. بارگذاری اسناد PDF با `PyPDFLoader`: چالش ساختار بصری

فرمت PDF برای **«چاپ شدن بر روی کاغذ»** طراحی شده است، نه برای خوانده شدن توسط کامپیوتر! در ساختار داخلی PDF چیزی به نام «پاراگراف» یا «جمله» وجود ندارد؛ بلکه موقعیت فیزیکی هر حرف به صورت مختصات $(x, y)$ روی صفحه ذخیره شده است.

لودر **`PyPDFLoader`** اسناد PDF را به ازای هر صفحه به یک شیء `Document` مجزا تبدیل می‌کند و **شماره صفحه** را در کلید `page` متادیتا قرار می‌دهد.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# بارگذاری فایل PDF محلی نمونه
pdf_path = "sample_paper.pdf"
pdf_loader = PyPDFLoader(pdf_path)

# بارگذاری کامل به صورت Eager
pdf_pages = pdf_loader.load()

print(f"تعداد صفحات بارگذاری‌شده از PDF: {len(pdf_pages)}\n")

for i, page in enumerate(pdf_pages):
    print(f"--- صفحه {i + 1} ---")
    print(f"متادیتا: {page.metadata}")
    print(f"محتوا:   {page.page_content.strip()}\n")

> 💡 **ارزش کلید `page` در متادیتا:**
> وجود شماره صفحه در متادیتا به چت‌بات‌های RAG این امکان را می‌دهد که در کنار ارائه پاسخ، دقیقاً به کاربر ارجاع دهند: *«منبع: مقاله هوش مصنوعی، صفحه ۲»*.
>
> 🌐 **پشتیبانی از URL مستقیم:**
> شیء `PyPDFLoader` به صورت خودکار قادر است لینک‌های آنلاین را نیز دانلود و پارس کند:
> ```python
> online_loader = PyPDFLoader("https://arxiv.org/pdf/2310.06825.pdf")
> online_pages = online_loader.load()
> ```
>
> ⚠️ **نکته خطایابی در PDFهای فارسی:**
> اگر فایل PDF شما فارسی و حاوی حروف جدا از هم یا متن اسکن‌شده (عکس) بود، باید از لودرهای مبتنی بر OCR نظیر `UnstructuredPDFLoader` استفاده کنید.

### ۳. بارگذاری صفحات وب با `WebBaseLoader`

صفحات وب سرشار از داده‌های زائد هستند: منوهای ناوبری، هدر، فوتر، تبلیغات، نظرات اسپم و کدهای جاوا اسکریپت.
اگر کل صفحه وب را بدون فیلتر وارد سیستم RAG کنید:
1. توکن‌های گران‌قیمت پرامپت با کلمات بی‌ربط تلف می‌شوند.
2. دقت پاسخگویی مدل به دلیل حواس‌پرتی و شلوغی کانتکست افت می‌کند.

> **نکته فنی مهم:** شیء `WebBaseLoader` منحصراً آدرس‌های اینترنتی با پروتکل HTTP یا HTTPS را می‌پذیرد (نه مسیر فایل محلی).
>
> برای جلوگیری از مسدود شدن درخواست‌ها توسط وب‌سایت‌ها و رفع اخطار، متغیر محیطی `USER_AGENT` را مقداردهی می‌کنیم.

In [ ]:
import os
import bs4
from langchain_community.document_loaders import WebBaseLoader

# تنظیم User-Agent برای جلوگیری از اخطار و شبیه‌سازی درخواست استاندارد مرورگر
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

target_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"

try:
    # --- روش ۱: بارگذاری همه چیز (کثیف و شامل منو و فوتر) ---
    loader_dirty = WebBaseLoader(target_url)
    docs_dirty = loader_dirty.load()
    print(f"طول متن (کثیف): {len(docs_dirty[0].page_content)} کاراکتر")

    # --- روش ۲: بارگذاری هدفمند (تمیز و حرفه‌ای) با SoupStrainer ---
    # فقط استخراج بخش‌های دارای کلاس‌های اصلی محتوا
    loader_clean = WebBaseLoader(
        target_url,
        bs_kwargs={
            "parse_only": bs4.SoupStrainer(
                class_=("post-content", "post-title", "post-header")
            )
        }
    )
    docs_clean = loader_clean.load()
    print(f"طول متن (تمیز): {len(docs_clean[0].page_content)} کاراکتر")
    print("\n--- بخشی از متن تمیز استخراج‌شده ---")
    print(docs_clean[0].page_content[:300].strip() + "...")

except Exception as e:
    print(f"⚠️ خطا در اتصال شبکه به وب‌سایت ({e})")
    print("\nدر صورت عدم دسترسی به اینترنت، مفهوم SoupStrainer را روی نمونه HTML زیر مشاهده کنید:")
    sample_html = '''
    <div class="navbar">منو، تبلیغات و پیوندها</div>
    <div class="post-title">مقدمه‌ای بر ایجنت‌های هوشمند در LLM</div>
    <div class="post-content">ایجنت‌ها سیستم‌هایی هستند که از LLM برای تصمیم‌گیری استفاده می‌کنند.</div>
    <div class="footer">فوتر سایت و کپی‌رایت</div>
    '''
    soup_all = bs4.BeautifulSoup(sample_html, 'html.parser')
    soup_target = bs4.BeautifulSoup(sample_html, 'html.parser', parse_only=bs4.SoupStrainer(class_=('post-title', 'post-content')))
    print(f"طول با تمام عناصر: {len(soup_all.get_text().strip())} کاراکتر")
    print(f"طول با فیلتر SoupStrainer: {len(soup_target.get_text().strip())} کاراکتر")
    print("متن تمیز:", soup_target.get_text().strip())

### ۴. رویکرد مهندسی پیشرفته: Eager Loading در برابر Lazy Loading

تاکنون از متد **`.load()`** استفاده می‌کردیم که به شیوه **مشتاق (Eager)** کار می‌کند:
1. تمام فایل‌ها را یکجا از دیسک می‌خواند.
2. تمام اسناد را همزمان در حافظه **RAM** لود می‌کند.
3. یک لیست کامل از اشیاء `Document` بازمی‌گرداند.

> 💥 **فاجعه در مقیاس سازمانی:**
> اگر بخواهید ۱۰,۰۰۰ کتاب یا گزارش سالانه را پردازش کنید، متد `.load()` باعث خطای **`MemoryError`** و کرش کردن سرور می‌شود!

**راهکار حرفه‌ای: متد `.lazy_load()`**
این متد یک **Generator** پایتونی برمی‌گرداند. اسناد یکی‌یکی از دیسک خوانده شده، پردازش می‌شوند و سپس از RAM تخلیه می‌گردند؛ بنابراین مصرف حافظه در طول کل فرآیند تقریباً **ثابت و مینیمال** باقی می‌ماند.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# راه‌اندازی لودر دایرکتوری جاری برای فایل‌های txt
dir_loader = DirectoryLoader(
    path=".",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

# ایجاد ژنراتور به شیوه Lazy Loading
docs_generator = dir_loader.lazy_load()

print(f"نوع داده برگشتی: {type(docs_generator)}")

# پردازش دانه به دانه اسناد بدون مصرف حافظه تجمیعی
print("\nشروع پردازش جریانی (Streaming Pipeline):")
for doc in docs_generator:
    source_file = doc.metadata.get("source", "نامشخص")
    char_count = len(doc.page_content)
    print(f"  📄 پردازش فایل: {source_file} (تعداد کاراکتر: {char_count})")

### 🎯 جمع‌بندی نکات کلیدی این بخش:

1. **استاندارد واحد لودرها:** تنوع فایل‌ها در LangChain با ده‌ها کلاس لودر پوشش داده شده و خروجی همه آن‌ها یکسان است (`List[Document]`).
2. **TextLoader:** ساده‌ترین راهکار برای فایل‌های متنی با تنظیم مهم `encoding="utf-8"`.
3. **PyPDFLoader:** تفکیک خودکار صفحات و ذخیره شماره صفحه در `metadata['page']` برای استناد دقیق در پاسخ‌ها.
4. **WebBaseLoader:** الزام پروتکل HTTP/HTTPS و استفاده هوشمندانه از `SoupStrainer` برای حذف منوها و تبلیغات و صرفه‌جویی چشمگیر در توکن و هزینه.
5. **Lazy Loading:** استفاده از `lazy_load()` در پروژه‌های حجیم برای پیشگیری از سربار حافظه (OOM) و پایداری سرورها.

**چالش پیش‌رو:**
اکنون اسناد را بارگذاری کردیم، اما برخی از این اسناد (مانند کتاب‌ها و مقالات) بسیار طولانی هستند و در پنجره زمینه مدل جا نمی‌شوند.
در درس‌نامه آینده یاد می‌گیریم که چگونه این اسناد را با **استراتژی‌های خرد کردن متن (Text Splitting)** بدون ناقص شدن معنای جملات قطعه‌قطعه کنیم.